# preTextAnalysis
Entrada: `acciones_seleccionadas.csv` (8 partidos, acciones ya seleccionadas a mano en "Economía y papel del Estado"). Salida: `acciones_preparadas.csv`, listo para `TextAnalysis`.

## preTEXT-01: leer el corpus
Lee el CSV directamente desde el repositorio del curso.

In [1]:
import pandas as pd

URL = "https://raw.githubusercontent.com/doctorado-cuanticp/text/main/acciones_seleccionadas.csv"
corpus = pd.read_csv(URL, keep_default_na=False)
corpus

,id_partido,partido,candidato,seccion,pagina,texto_original
0,A,Renovación Popular Social,Candidata A,Economía y papel del Estado,1,Ampliaremos las empresas públicas en sectores ...
1,B,Frente Progresista,Candidato B,Economía y papel del Estado,1,Fortaleceremos la gestión de las empresas públ...
2,C,Movimiento Democrático,Candidata C,Economía y papel del Estado,1,Reduciremos la evasión y revisaremos las exone...
3,D,Centro Cívico,Candidato D,Economía y papel del Estado,1,Exigiremos a las empresas públicas metas verif...
4,E,Alianza Nacional,Candidata E,Economía y papel del Estado,1,Reestructuraremos las empresas públicas defici...
5,F,Libertad Republicana,Candidato F,Economía y papel del Estado,1,Incorporaremos capital privado a empresas públ...
6,G,Futuro Liberal,Candidata G,Economía y papel del Estado,1,Transferiremos a operadores privados las empre...
7,H,Mercado y Libertad,Candidato H,Economía y papel del Estado,1,Privatizaremos las empresas públicas productiv...


## preTEXT-02: estructura
Ocho filas, una por partido. Se registra `num_propuestas` (varía entre 4 y 9) porque afecta la interpretación en `TextAnalysis`.

In [2]:
print('Total de partidos:', len(corpus))

corpus['num_propuestas'] = corpus['texto_original'].apply(
    lambda t: len([l for l in str(t).split('\n') if l.strip()])
)
corpus[['id_partido', 'partido', 'seccion', 'pagina', 'num_propuestas']]

Total de partidos: 8


,id_partido,partido,seccion,pagina,num_propuestas
0,A,Renovación Popular Social,Economía y papel del Estado,1,5
1,B,Frente Progresista,Economía y papel del Estado,1,7
2,C,Movimiento Democrático,Economía y papel del Estado,1,4
3,D,Centro Cívico,Economía y papel del Estado,1,6
4,E,Alianza Nacional,Economía y papel del Estado,1,8
5,F,Libertad Republicana,Economía y papel del Estado,1,5
6,G,Futuro Liberal,Economía y papel del Estado,1,9
7,H,Mercado y Libertad,Economía y papel del Estado,1,6


## preTEXT-03: preparar el texto
Normaliza espacios y acentos; conserva negaciones, condiciones y puntuación.

In [3]:
import re, unicodedata

def limpiar(texto):
    texto = unicodedata.normalize('NFC', texto)
    texto = texto.replace('\u00ad', '')
    return re.sub(r'\s+', ' ', texto).strip()

corpus['texto'] = corpus['texto_original'].apply(limpiar)
corpus[['id_partido', 'texto']]

,id_partido,texto
0,A,Ampliaremos las empresas públicas en sectores ...
1,B,Fortaleceremos la gestión de las empresas públ...
2,C,Reduciremos la evasión y revisaremos las exone...
3,D,Exigiremos a las empresas públicas metas verif...
4,E,Reestructuraremos las empresas públicas defici...
5,F,Incorporaremos capital privado a empresas públ...
6,G,Transferiremos a operadores privados las empre...
7,H,Privatizaremos las empresas públicas productiv...


## preTEXT-04: comparar con el original
Confirma que la limpieza no alteró contenido. No valida la selección manual — eso exige volver al PDF.

In [4]:
for _, fila in corpus.iterrows():
    print(f"\n{fila['id_partido']} — {fila['partido']}")
    print('ORIGINAL :', fila['texto_original'][:150], '...')
    print('PREPARADO:', fila['texto'][:150], '...')


A — Renovación Popular Social
ORIGINAL : Ampliaremos las empresas públicas en sectores estratégicos y destinaremos sus utilidades a inversión productiva y servicios esenciales.
Aplicaremos un ...
PREPARADO: Ampliaremos las empresas públicas en sectores estratégicos y destinaremos sus utilidades a inversión productiva y servicios esenciales. Aplicaremos un ...

B — Frente Progresista
ORIGINAL : Fortaleceremos la gestión de las empresas públicas estratégicas mediante directorios profesionales, metas de servicio y rendición de cuentas.
Aumentar ...
PREPARADO: Fortaleceremos la gestión de las empresas públicas estratégicas mediante directorios profesionales, metas de servicio y rendición de cuentas. Aumentar ...

C — Movimiento Democrático
ORIGINAL : Reduciremos la evasión y revisaremos las exoneraciones tributarias para financiar inversión sin comprometer el equilibrio fiscal.
Financiaremos capaci ...
PREPARADO: Reduciremos la evasión y revisaremos las exoneraciones tributarias para fina

## preTEXT-05: guardar
Exporta el corpus final, una fila por partido.

In [5]:
assert len(corpus) > 0 and corpus['texto'].ne('').all(), 'Complete los textos antes de guardar.'
assert corpus['id_partido'].is_unique, 'Debe haber una sola fila por partido.'

corpus.to_csv('acciones_preparadas.csv', index=False, encoding='utf-8')
print('Guardado: acciones_preparadas.csv —', len(corpus), 'partidos.')

Guardado: acciones_preparadas.csv — 8 partidos.
